In [1]:
# Phase 2 – Data Engineering | Cell 1: Setup + Load Phase 1

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")

OUT_DIR  = PROJECT / "outputs" / "phase2"
OUT_DATA = OUT_DIR / "data"
OUT_TAB  = OUT_DIR / "tables"
OUT_FIG  = OUT_DIR / "figures"
OUT_LOG  = OUT_DIR / "logs"

for d in [OUT_DATA, OUT_TAB, OUT_FIG, OUT_LOG]:
    d.mkdir(parents=True, exist_ok=True)

P1_DATA = PROJECT / "outputs" / "phase1" / "data"

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 1")
print("=" * 60)
print(f"Timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root : {PROJECT}")
print(f"Outputs      : {OUT_DIR}")

jst  = pd.read_pickle(P1_DATA / "jst_core_2000_2020.pkl")
baci = pd.read_pickle(P1_DATA / "baci_edges_jst18.pkl")
nodes = pd.read_pickle(P1_DATA / "baci_nodes_jst18.pkl")
bis  = pd.read_pickle(P1_DATA / "bis_credit_panel.pkl")
cw   = pd.read_pickle(P1_DATA / "country_crosswalk.pkl")

print("\nLoaded from Phase 1:")
print(f"  JST core   : {jst.shape}")
print(f"  BACI edges : {baci.shape}")
print(f"  BACI nodes : {nodes.shape}")
print(f"  BIS panel  : {bis.shape}")
print(f"  Crosswalk  : {cw.shape}")

print("\nCell 1 completed successfully.")

Phase 2 – Data Engineering | Cell 1
Timestamp    : 2026-09-25 20:26:02
Project Root : D:\univercity\omid_project\economics_project
Outputs      : D:\univercity\omid_project\economics_project\outputs\phase2

Loaded from Phase 1:
  JST core   : (378, 62)
  BACI edges : (6426, 14)
  BACI nodes : (378, 8)
  BIS panel  : (378, 4)
  Crosswalk  : (18, 7)

Cell 1 completed successfully.


In [2]:
# Phase 2 – Data Engineering | Cell 2: JST Cleaning + Missing Analysis

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"
P1_DATA  = PROJECT / "outputs" / "phase1" / "data"

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 2: JST Cleaning")
print("=" * 60)

jst = pd.read_pickle(P1_DATA / "jst_core_2000_2020.pkl").copy()

print("--- Basic info ---")
print(f"Shape: {jst.shape}")
print(f"Duplicated rows: {jst.duplicated().sum()}")
jst = jst.drop_duplicates()
print(f"Shape after drop duplicates: {jst.shape}")

# Core macro columns for analysis
core_cols = [
    "year", "country", "iso2", "iso3",
    "pop", "rgdpmad", "rgdpbarro", "gdp", "iy", "cpi",
    "stir", "ltrate", "money", "narrowm", "unemp", "wage",
    "debtgdp", "revenue", "expenditure",
    "tloans", "tmort", "thh", "tbus", "bdebt",
    "crisisJST", "ca", "imports", "exports"
]
core_cols = [c for c in core_cols if c in jst.columns]
jst_core = jst[core_cols].copy()

print(f"\nCore columns used: {len(core_cols)}")
print(core_cols)

# Missing report
miss = pd.DataFrame({
    "column": jst_core.columns,
    "missing_count": jst_core.isna().sum().values,
    "missing_pct": (jst_core.isna().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print("\n--- Missing values (top) ---")
print(miss.head(15).to_string(index=False))

miss.to_csv(OUT_TAB / "jst_missing_report.csv", index=False)
jst_core.to_pickle(OUT_DATA / "jst_core_step1.pkl")

print(f"\nSaved → {OUT_DATA / 'jst_core_step1.pkl'}")
print("\nCell 2 completed successfully.")

Phase 2 – Data Engineering | Cell 2: JST Cleaning
--- Basic info ---
Shape: (378, 62)
Duplicated rows: 0
Shape after drop duplicates: (378, 62)

Core columns used: 28
['year', 'country', 'iso2', 'iso3', 'pop', 'rgdpmad', 'rgdpbarro', 'gdp', 'iy', 'cpi', 'stir', 'ltrate', 'money', 'narrowm', 'unemp', 'wage', 'debtgdp', 'revenue', 'expenditure', 'tloans', 'tmort', 'thh', 'tbus', 'bdebt', 'crisisJST', 'ca', 'imports', 'exports']

--- Missing values (top) ---
     column  missing_count  missing_pct
      tmort              4         1.06
     tloans              3         0.79
  rgdpbarro              2         0.53
expenditure              2         0.53
    revenue              2         0.53
       year              0         0.00
       wage              0         0.00
    imports              0         0.00
         ca              0         0.00
  crisisJST              0         0.00
      bdebt              0         0.00
       tbus              0         0.00
        thh         

In [3]:
# Phase 2 – Data Engineering | Cell 3: Treat missing + Merge + Outliers

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"
P1_DATA  = PROJECT / "outputs" / "phase1" / "data"

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 3: Merge + Outliers")
print("=" * 60)

jst = pd.read_pickle(OUT_DATA / "jst_core_step1.pkl").copy()
bis = pd.read_pickle(P1_DATA / "bis_credit_panel.pkl").copy()
nodes = pd.read_pickle(P1_DATA / "baci_nodes_jst18.pkl").copy()

# --- Light missing treatment (group-wise interpolate by country) ---
num_cols = jst.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ["year", "crisisJST"]]

jst = jst.sort_values(["country", "year"])
for c in num_cols:
    jst[c] = jst.groupby("country")[c].transform(
        lambda s: s.interpolate(method="linear", limit_direction="both")
    )

# residual fill with country median
for c in num_cols:
    jst[c] = jst.groupby("country")[c].transform(lambda s: s.fillna(s.median()))

print("Missing after treatment:")
print(jst[num_cols].isna().sum().sort_values(ascending=False).head(8))

# --- Merge BIS ---
panel = jst.merge(bis, on=["year", "iso2"], how="left")
print(f"\nAfter BIS merge: {panel.shape}")

# --- Merge BACI node attrs ---
nodes = nodes.rename(columns={"country": "iso2"})
panel = panel.merge(
    nodes[["year", "iso2", "export_kusd", "import_kusd", "net_trade_kusd", "trade_openness_kusd"]],
    on=["year", "iso2"], how="left"
)
print(f"After BACI nodes merge: {panel.shape}")

# --- Outlier detection (IQR x 3) on key vars ---
key_vars = [c for c in [
    "rgdpmad", "gdp", "cpi", "stir", "ltrate", "debtgdp", "unemp", "iy",
    "money", "tloans", "revenue", "expenditure", "ca", "imports", "exports",
    "credit_private_pct_gdp", "credit_gov_pct_gdp",
    "export_kusd", "import_kusd", "trade_openness_kusd"
] if c in panel.columns]

outlier_rows = []
for col in key_vars:
    s = panel[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 3 * iqr, q3 + 3 * iqr
    mask = (panel[col] < low) | (panel[col] > high)
    n = int(mask.sum())
    outlier_rows.append({
        "column": col,
        "outliers_count": n,
        "outliers_pct": round(100 * n / len(panel), 2),
        "low": low, "high": high
    })
    # flag only (do not drop rows)
    panel[f"{col}_outlier"] = mask.astype(int)

out_df = pd.DataFrame(outlier_rows).sort_values("outliers_pct", ascending=False)
print("\n--- Outlier summary (IQR x 3) ---")
print(out_df.head(12).to_string(index=False))

out_df.to_csv(OUT_TAB / "outlier_report.csv", index=False)
panel.to_pickle(OUT_DATA / "panel_merged_step2.pkl")
panel.to_csv(OUT_TAB / "panel_merged_step2.csv", index=False)

print(f"\nPanel shape: {panel.shape}")
print(f"Saved → {OUT_DATA / 'panel_merged_step2.pkl'}")
print("\nCell 3 completed successfully.")

Phase 2 – Data Engineering | Cell 3: Merge + Outliers
Missing after treatment:
pop        0
debtgdp    0
imports    0
ca         0
bdebt      0
tbus       0
thh        0
tmort      0
dtype: int64

After BIS merge: (378, 30)
After BACI nodes merge: (378, 34)

--- Outlier summary (IQR x 3) ---
             column  outliers_count  outliers_pct           low         high
            exports              63         16.67 -2.798142e+06 3.734217e+06
            imports              63         16.67 -2.512712e+06 3.353746e+06
              money              63         16.67 -6.416651e+06 8.564862e+06
                gdp              58         15.34 -9.343347e+06 1.246965e+07
             tloans              56         14.81 -8.635578e+06 1.152486e+07
        expenditure              55         14.55 -2.903057e+06 3.872651e+06
            revenue              54         14.29 -2.865059e+06 3.821903e+06
                 ca              54         14.29 -2.087134e+05 2.773143e+05
        import

In [4]:
# Phase 2 – Data Engineering | Cell 4: Stationarity Tests (ADF + KPSS)

from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 4: Stationarity")
print("=" * 60)

panel = pd.read_pickle(OUT_DATA / "panel_merged_step2.pkl")

vars_test = [c for c in [
    "rgdpmad", "gdp", "cpi", "stir", "ltrate", "debtgdp", "unemp", "iy",
    "money", "tloans", "revenue", "expenditure", "ca",
    "credit_private_pct_gdp", "credit_gov_pct_gdp",
    "export_kusd", "import_kusd", "trade_openness_kusd"
] if c in panel.columns]

records = []
for country, g in panel.groupby("country"):
    g = g.sort_values("year")
    for var in vars_test:
        s = g[var].dropna()
        if len(s) < 8:
            continue
        # ADF
        try:
            adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
            adf_stat_flag = adf_p < 0.05
        except Exception:
            adf_stat, adf_p, adf_stat_flag = np.nan, np.nan, False
        # KPSS
        try:
            kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
            kpss_stat_flag = kpss_p > 0.05
        except Exception:
            kpss_stat, kpss_p, kpss_stat_flag = np.nan, np.nan, False

        records.append({
            "country": country,
            "variable": var,
            "n_obs": len(s),
            "ADF_stat": round(float(adf_stat), 4) if pd.notna(adf_stat) else np.nan,
            "ADF_pvalue": round(float(adf_p), 4) if pd.notna(adf_p) else np.nan,
            "KPSS_stat": round(float(kpss_stat), 4) if pd.notna(kpss_stat) else np.nan,
            "KPSS_pvalue": round(float(kpss_p), 4) if pd.notna(kpss_p) else np.nan,
            "ADF_stationary": bool(adf_stat_flag),
            "KPSS_stationary": bool(kpss_stat_flag),
        })

res = pd.DataFrame(records)
print("--- Sample results ---")
print(res.head(12).to_string(index=False))

summary = res.groupby("variable").agg(
    countries_tested=("country", "count"),
    ADF_stationary_pct=("ADF_stationary", lambda x: round(100 * x.mean(), 1)),
    KPSS_stationary_pct=("KPSS_stationary", lambda x: round(100 * x.mean(), 1)),
).reset_index()
print("\n--- Summary by variable ---")
print(summary.to_string(index=False))

res.to_csv(OUT_TAB / "stationarity_tests_detailed.csv", index=False)
summary.to_csv(OUT_TAB / "stationarity_summary_by_variable.csv", index=False)

print(f"\nSaved detailed + summary tables")
print("\nCell 4 completed successfully.")

Phase 2 – Data Engineering | Cell 4: Stationarity


C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outsid

--- Sample results ---
  country    variable  n_obs  ADF_stat  ADF_pvalue  KPSS_stat  KPSS_pvalue  ADF_stationary  KPSS_stationary
Australia     rgdpmad     21   -2.8063      0.0574     0.7690       0.0100           False            False
Australia         gdp     21    0.3741      0.9805     0.7994       0.0100           False            False
Australia         cpi     21   -1.7005      0.4309     0.8006       0.0100           False            False
Australia        stir     21   -2.6393      0.0851     0.6991       0.0136           False            False
Australia      ltrate     21    2.0199      0.9987     0.7529       0.0100           False            False
Australia     debtgdp     21    2.5941      0.9991     0.6992       0.0136           False            False
Australia       unemp     21   -1.7949      0.3830     0.1445       0.1000           False             True
Australia          iy     21   -0.7519      0.8328     0.2747       0.1000           False             True
Austr

C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned.
  kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
C:\Users\maka\AppData\Local\Temp\ipykernel_3692\361675564.py:40: InterpolationWarning: The test statistic is outsid

In [5]:
# Phase 2 – Data Engineering | Cell 5: Differencing + Final Clean Panel

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase2" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase2" / "tables"

print("=" * 60)
print("Phase 2 – Data Engineering | Cell 5: Final Panel")
print("=" * 60)

panel = pd.read_pickle(OUT_DATA / "panel_merged_step2.pkl").copy()
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

# Variables to difference / log-difference
level_vars = [c for c in [
    "rgdpmad", "gdp", "cpi", "money", "narrowm", "tloans", "tmort",
    "revenue", "expenditure", "imports", "exports",
    "export_kusd", "import_kusd", "trade_openness_kusd",
    "credit_private_pct_gdp", "credit_gov_pct_gdp",
    "debtgdp", "unemp", "iy", "stir", "ltrate", "ca"
] if c in panel.columns]

# already ratio-like / rates: still add first difference
for v in level_vars:
    panel[f"{v}_diff"] = panel.groupby("country")[v].diff()
    # log-diff only for strictly positive vars
    if panel[v].gt(0).mean() > 0.8:
        panel[f"{v}_logdiff"] = panel.groupby("country")[v].transform(
            lambda s: np.log(s.replace(0, np.nan)).diff()
        )

# Drop pure outlier flag clutter from final analytic set (keep data)
flag_cols = [c for c in panel.columns if c.endswith("_outlier")]
panel_final = panel.drop(columns=flag_cols)

print(f"Added differenced columns for {len(level_vars)} variables.")
print(f"Final panel shape: {panel_final.shape}")

print("\nTop missing in final panel:")
print(panel_final.isna().mean().sort_values(ascending=False).head(10).round(3))

panel_final.to_pickle(OUT_DATA / "panel_final_clean.pkl")
panel_final.to_csv(OUT_TAB / "panel_final_clean.csv", index=False)

# Compact feature list for next phases
meta = pd.DataFrame({
    "n_rows": [len(panel_final)],
    "n_cols": [panel_final.shape[1]],
    "n_countries": [panel_final["iso3"].nunique()],
    "year_min": [int(panel_final["year"].min())],
    "year_max": [int(panel_final["year"].max())],
})
meta.to_csv(OUT_TAB / "panel_final_meta.csv", index=False)
print(meta.to_string(index=False))

print(f"\nSaved → {OUT_DATA / 'panel_final_clean.pkl'}")
print("\nCell 5 completed successfully.")
print("PHASE 2 CORE COMPLETED.")

Phase 2 – Data Engineering | Cell 5: Final Panel
Added differenced columns for 22 variables.
Final panel shape: (378, 75)

Top missing in final panel:
credit_gov_pct_gdp_diff           0.471
credit_gov_pct_gdp                0.444
credit_private_pct_gdp_diff       0.206
credit_private_pct_gdp_logdiff    0.206
credit_private_pct_gdp            0.167
ltrate_logdiff                    0.103
imports_diff                      0.048
expenditure_logdiff               0.048
expenditure_diff                  0.048
revenue_logdiff                   0.048
dtype: float64
 n_rows  n_cols  n_countries  year_min  year_max
    378      75           18      2000      2020

Saved → D:\univercity\omid_project\economics_project\outputs\phase2\data\panel_final_clean.pkl

Cell 5 completed successfully.
PHASE 2 CORE COMPLETED.
